## Multiclass Classification

### Objectives

After completing this session, you will be able to:

1. Understand the use of one-hot encoding for categorical variable
2. Implement logistic regression for multi-class classification using **One-vs-All (OvA)** and **One-vs-One (OvO)** strategies.
3. Evaluate model performance using appropriate metrics.

## Machine Learning

Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed. Broadly divided into 3 main types:

1. Supervised Learning

Supervised learning uses labeled data, meaning the correct output is already known during training. The algorithm learns the relationship between input variables and the output.

Common tasks include:

- Classification – predicting categories (e.g., disease type, spam detection). We touched a little of this last week, Logistic regression.

- Regression – predicting continuous values (e.g., blood pressure level).

Multiclass classification belongs to this category.

2. Unsupervised Learning

Unsupervised learning works with unlabeled data, where the algorithm tries to discover patterns or structures within the data.

Common tasks include:

- Clustering – grouping similar observations.

- Dimensionality reduction – reducing the number of variables while preserving information.

3. Reinforcement Learning

Reinforcement learning involves an agent learning by interacting with an environment. The model receives rewards or penalties based on its actions and learns strategies to maximize rewards over time.

Applications include:

- Robotics

- Game playing

- Autonomous systems

### Classification vs Regression

**Classification** is used when the outcome variable is categorical. The goal is to assign observations into predefined classes or categories. For example, a model may classify an email as spam or not spam, predict whether a patient is disease positive or negative, or categorize disease severity as mild, moderate, or severe. Classification can be binary (two classes) or multiclass (more than two classes).

**Regression**, is used when the outcome variable is continuous or numerical. The aim is to predict a quantitative value rather than a category. Examples include predicting blood pressure level, patient survival time, hospital cost, or temperature.


## Classification

This can be either **Binary** or **Multi-class** (Depending on their output values)

- **Binary**, the outcome has only $2$ **possible values**:

$$y \in \{0,1\}$$

The outcome can not be modelled directly but the probability of that an observation belongs to class $1$ can be modelled. E. g

$$P(y = 1\mid X) = \frac{1}{1+ e^{-(\beta_0 + \beta_1 x_1 + ... + \beta_p x_p)}}$$

The decision rule can be:
$$
\hat{y} =
\begin{cases}
1 & \text{if } P(y=1 \mid X) > \delta \\
0 & \text{otherwise}
\end{cases}
$$

where $\delta$ is threshold, sometimes $0.5$


While for **Multi-class**, the response variable can take **more than $2$ categories**:

$$y\in \{1,2,3,...,K\}$$
where $K$ is the number of classes.

Just like in logistic regression, the outcome cannot be modelled directly, but the probability that an observation belongs to each class:

$$P(y = k \mid X) = \frac{e^{\beta_k^TX}}{\sum_{j=1}^K e^{\beta_j^TX}}$$

Decision rule:

$$\hat{y} = \arg\max_{k} P(y = k\mid X)$$


## Logistic Model with One-vs-All and One-vs-One 

Multi-class Logistic Model has 2 methods of classifying observations into categories:


### Logistic Regression with One-vs-All (OvA)
In the One-vs-All approach:

* The algorithm trains a single binary classifier for each class.
* Each classifier learns to distinguish a single class from all the others combined.
* If there are k classes, k classifiers are trained.
* During prediction, the algorithm evaluates all classifiers on each input, and selects the class with the highest confidence score as the predicted class.

#### Advantages:
* Simpler and more efficient in terms of the number of classifiers (k)
* Easier to implement for algorithms that naturally provide confidence scores (e.g., logistic regression, SVM).

#### Disadvantages:
* Classifiers may struggle with class imbalance since each binary classifier must distinguish between one class and the rest.
* Requires the classifier to perform well even with highly imbalanced datasets, as the "all" group typically contains more samples than the "one" class.


### Logistic Regression with OvO

In the One-vs-One approach:
* The algorithm trains a binary classifier for every pair of classes in the dataset.
* If there are k classes, this results in $k(k-1)/2$ classifiers.
* Each classifier is trained to distinguish between two specific classes, ignoring the rest.
* During prediction, all classifiers are used, and a "voting" mechanism decides the final class by selecting the class that wins the majority of pairwise comparisons.

#### Advantages:
* Suitable for algorithms that are computationally expensive to train on many samples because each binary classifier deals with a smaller dataset (only samples from two classes).
* Can be more accurate in some cases since classifiers focus on distinguishing between two specific classes at a time.

#### Disadvantages:
* Computationally expensive for datasets with a large number of classes due to the large number of classifiers required.
* May lead to ambiguous predictions if voting results in a tie.




## Practical

We have touched Binary logistic regression last week, we proceed to multi-class classification.

The data is from [UCI library](https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition), and we will use it under the [CCA 4.0](https://creativecommons.org/licenses/by/4.0/legalcode) license. The data set has 17 attributes in total along with 2111 samples. 

| Variable Name | Type | Description |
|---------------|------|-------------|
| Gender | Categorical | |
| Age | Continuous | |
| Height | Continuous | |
| Weight | Continuous | |
| family_history_with_overweight | Binary | Has a family member suffered or suffers from overweight? |
| FAVC | Binary | Do you eat high caloric food frequently? |
| FCVC | Integer | Do you usually eat vegetables in your meals? |
| NCP | Continuous | How many main meals do you have daily? |
| CAEC | Categorical | Do you eat any food between meals? |
| SMOKE | Binary | Do you smoke? |
| CH2O | Continuous | How much water do you drink daily? |
| SCC | Binary | Do you monitor the calories you eat daily? |
| FAF | Continuous | How often do you have physical activity? |
| TUE | Integer | Time spent using technological devices |
| CALC | Categorical | How often do you drink alcohol? |
| MTRANS | Categorical | Transportation usually used |
| NObeyesdad | Categorical | Obesity level |

## Load necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
file_path = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/GkDzb7bWrtvGXdPOfk6CIg/Obesity-level-prediction-dataset.csv"
data = pd.read_csv(file_path)
data.head()

## Exploratory Data Analysis (EDA)


Examine how balanced the data is

In [ ]:
sns.countplot(y='NObeyesdad', data=data)
plt.title('Distribution of Obesity Levels')
plt.show()

Check for null values, and display a summmary of the dataset

In [ ]:
data.isnull().sum()


In [ ]:
data.info()
data.describe()


## Data Preprocessing

1. Feature Scaling: Perform this for all numeric variables. This standardizes the values and improve the model performance.

In [ ]:
# Standardizing continuous numerical features
continuous_columns = data.select_dtypes(include=['float64']).columns.tolist()

scaler = StandardScaler()
scaled_features = scaler.fit_transform(data[continuous_columns])

# Converting to a DataFrame
scaled_df = pd.DataFrame(scaled_features, columns=scaler.get_feature_names_out(continuous_columns))

# Combining with the original dataset
scaled_data = pd.concat([data.drop(columns=continuous_columns), scaled_df], axis=1)

scaled_data.head()

2. One Hot Encoding

Convert categorical variables into numerical format using one-hot encoding.



In [ ]:
# Identifying categorical columns
categorical_columns = scaled_data.select_dtypes(include=['object']).columns.tolist()
categorical_columns.remove('NObeyesdad')  # Exclude target column

# Applying one-hot encoding
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_features = encoder.fit_transform(scaled_data[categorical_columns])

# Converting to a DataFrame
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_columns))

# Combining with the original dataset
prepped_data = pd.concat([scaled_data.drop(columns=categorical_columns), encoded_df], axis=1)

prepped_data.head()

3. Encode the target variable

In [ ]:
prepped_data['NObeyesdad'] = prepped_data['NObeyesdad'].astype('category').cat.codes
prepped_data.head()

# Preparing final dataset
X = prepped_data.drop('NObeyesdad', axis=1)
y = prepped_data['NObeyesdad']

## Model Training and Evaluation

### Data Splitting

Train and Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#### One-vs-All (OvA)

In [ ]:
model_ova = OneVsRestClassifier(LogisticRegression(max_iter=1000)) # here we use OneVsRestClassifier to implement the One-vs-Rest strategy for multiclass classification
model_ova.fit(X_train, y_train)

#### Model Evaluation

In [ ]:
y_pred_ova = model_ova.predict(X_test)

# Evaluation metrics for OvA
print("One-vs-All (OvA) Strategy")
print(f"Accuracy: {np.round(100*accuracy_score(y_test, y_pred_ova),2)}%")

#### One-vs-One (OvO)

In [ ]:
model_ovo = OneVsOneClassifier(LogisticRegression(max_iter=1000))
model_ovo.fit(X_train, y_train)

#### Evaluation

In [ ]:
y_pred_ovo = model_ovo.predict(X_test)

# Evaluation metrics for OvO
print("One-vs-One (OvO) Strategy")
print(f"Accuracy: {np.round(100*accuracy_score(y_test, y_pred_ovo),2)}%")

#### Improving Model performance

Experiment with different test sizes in the train_test_split method (e.g., 0.1, 0.3) and observe the impact on model performance.



In [ ]:
for state in [0.1, 0.3]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=state, random_state=4, stratify=y)
    model_ovo = OneVsOneClassifier(LogisticRegression(max_iter=1000))
    model_ovo.fit(X_train, y_train)
    y_pred_ovo = model_ovo.predict(X_test)
    # Evaluation metrics for OvO
    print(F"One-vs-One (OvO) Strategy for state being {state}")
    print(f"Accuracy: {np.round(100*accuracy_score(y_test, y_pred_ovo),2)}%")

In [ ]:
# OVA generally performs better with smaller test sizes, while OvO can be more robust with larger test sizes, but this can vary based on the specific dataset and class distribution.

for state in [0.1, 0.3]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=state, random_state=4, stratify=y)
    model_ova = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    model_ova.fit(X_train, y_train)
    y_pred_ova = model_ova.predict(X_test)
    # Evaluation metrics for OvA
    print(F"One-vs-all (OvA) Strategy for state = {state}")
    print(f"Accuracy: {np.round(100*accuracy_score(y_test, y_pred_ova),2)}%")

#### Variable Importance


In [ ]:
# For One vs Rest model
# Collect all coefficients from each underlying binary classifier
coefs = np.array([est.coef_[0] for est in model_ova.estimators_])

# Now take the mean across all those classifiers
feature_importance = np.mean(np.abs(coefs), axis=0)
plt.barh(X.columns, feature_importance)
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.show()

# For One vs One model
# Collect all coefficients from each underlying binary classifier
coefs = np.array([est.coef_[0] for est in model_ovo.estimators_])

# Now take the mean across all those classifiers
feature_importance = np.mean(np.abs(coefs), axis=0)

# Plot feature importance
plt.barh(X.columns, feature_importance)
plt.title("Feature Importance (One-vs-One)")
plt.xlabel("Importance")
plt.show()

Write a function `obesity_risk_pipeline` to automate the entire pipeline:

* Loading and preprocessing the data
* Training the model
* Evaluating the model

The function should accept the file path and test set size as the input arguments.



In [ ]:
def obesity_risk_pipeline(data_path, test_size=0.4):
    data = pd.read_csv(data_path)
    continuous_columns = data.select_dtypes(include=['float64']).columns.tolist()
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(data[continuous_columns])
    scaled_df = pd.DataFrame(scaled_features, columns=scaler.get_feature_names_out(continuous_columns))
    scaled_data = pd.concat([data.drop(columns=continuous_columns), scaled_df], axis=1)
    categorical_columns = scaled_data.select_dtypes(include=['object']).columns.tolist()
    categorical_columns.remove('NObeyesdad')
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    encoded_features = encoder.fit_transform(scaled_data[categorical_columns])
    encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_columns))
    prepped_data = pd.concat([scaled_data.drop(columns=categorical_columns), encoded_df], axis=1)
    prepped_data['NObeyesdad'] = prepped_data['NObeyesdad'].astype('category').cat.codes
    X = prepped_data.drop('NObeyesdad', axis=1)
    y = prepped_data['NObeyesdad']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    model_ova = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    model_ova.fit(X_train, y_train)
    y_pred_ova = model_ova.predict(X_test)
    print("One-vs-All (OvA) Strategy")
    print(f"Accuracy: {np.round(100*accuracy_score(y_test, y_pred_ova),2)}%")
    model_ovo = OneVsOneClassifier(LogisticRegression(max_iter=1000))
    model_ovo.fit(X_train, y_train)
    y_pred_ovo = model_ovo.predict(X_test)
    print("One-vs-One (OvO) Strategy")
    print(f"Accuracy: {np.round(100*accuracy_score(y_test, y_pred_ovo),2)}%")

obesity_risk_pipeline(file_path, test_size=0.2)

## Assignment

From the figures above, there are features that are of less importance to the models, train **OvA** and **OvO** models on a new dataset with only important features.